# FBRef Schedule Scraper

Scrapes fixture lists (date, teams, score, competition) for PL-relevant competitions.
Much lighter than the full match stats scraper — hits schedule pages only, no Selenium needed.

**Output:** `infra/data/json/fbref/fixtures_flat.csv` — same schema as the API-Football version so they can be concatenated.

**Rate limiting:** FBRef allows ~1 req/6s before soft-blocking. The scraper enforces this.

## 1. Config

In [1]:
import json
import time
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

COMPETITIONS = {
    "Premier League":                {"id": 9,   "name": "Premier-League"},
    "UEFA Champions League":         {"id": 8,   "name": "Champions-League"},
    "UEFA Europa League":            {"id": 19,  "name": "Europa-League"},
    "UEFA Europa Conference League": {"id": 882, "name": "Conference-League"},
    "FA Cup":                        {"id": 15,  "name": "FA-Cup"},
    "League Cup":                    {"id": 25,  "name": "EFL-Cup"},
}

SEASONS = [
    "2017-2018",
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
]

OUTPUT_DIR = Path("../../../../infra/data/json/fbref/schedules_raw").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FLAT_CSV = OUTPUT_DIR.parent / "fixtures_flat.csv"

REQUEST_DELAY = 8

print(f"Output: {OUTPUT_DIR}")
print(f"Total requests (uncached): {len(COMPETITIONS) * len(SEASONS)}")
print(f"Estimated time: ~{len(COMPETITIONS) * len(SEASONS) * REQUEST_DELAY / 60:.0f} mins")


Output: /Users/admin/dev/algobetting/infra/data/json/fbref/schedules_raw
Total requests (uncached): 30
Estimated time: ~4 mins


## 2. Scraper

In [2]:
import undetected_chromedriver as uc

CHROME_BINARY = (
    '/Users/admin/.cache/selenium/chrome/mac-arm64/'
    '147.0.7727.56/Google Chrome for Testing.app/'
    'Contents/MacOS/Google Chrome for Testing'
)

options = uc.ChromeOptions()
options.binary_location = CHROME_BINARY
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = uc.Chrome(options=options)
print("Browser started")


Browser started


In [3]:
# Debug: check what the browser actually sees
import time
driver.get('https://fbref.com/en/comps/9/2017-2018/schedule/2017-2018-Premier-League-Scores-and-Fixtures')
time.sleep(10)  # wait for Cloudflare
print('Title:', driver.title)
print('URL:', driver.current_url)
# Check if Cloudflare challenge is present
if 'Just a moment' in driver.title or 'Attention Required' in driver.title:
    print('BLOCKED: Cloudflare challenge not solved in headless mode')
else:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    tables = soup.find_all('table')
    print(f'Tables found: {len(tables)}')
    for t in tables:
        print(' -', t.get('id', 'no-id'))


Title: Just a moment...
URL: https://fbref.com/en/comps/9/2017-2018/schedule/2017-2018-Premier-League-Scores-and-Fixtures
BLOCKED: Cloudflare challenge not solved in headless mode


## 3. Fetch all competitions × seasons

In [4]:
all_rows = []
total = len(COMPETITIONS) * len(SEASONS)
done = 0

for comp_name, comp_info in COMPETITIONS.items():
    for season in SEASONS:
        done += 1
        print(f"[{done}/{total}]")
        rows = fetch_schedule(comp_name, comp_info["id"], comp_info["name"], season)
        all_rows.extend(rows)

print(f"\nTotal fixtures collected: {len(all_rows)}")

[1/30]


NameError: name 'fetch_schedule' is not defined

## 4. Parse scores and flatten to CSV

In [ ]:
def parse_score(score_str):
    """Parse '2–1' or '2-1' into (home_goals, away_goals)."""
    for sep in ["\u2013", "–", "-"]:
        if sep in score_str:
            parts = score_str.split(sep)
            try:
                return int(parts[0].strip()), int(parts[1].strip())
            except ValueError:
                return None, None
    return None, None

df = pd.DataFrame(all_rows)
df[["home_goals", "away_goals"]] = df["score"].apply(
    lambda s: pd.Series(parse_score(s))
)
df = df.dropna(subset=["home_goals", "away_goals"])

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df = df.sort_values("date").reset_index(drop=True)

# Keep only columns that match the API-Football output
df = df[["date", "season", "league_name", "round", "home_team", "away_team",
         "home_goals", "away_goals", "venue"]]

df.to_csv(FLAT_CSV, index=False)
print(f"Saved {len(df)} rows → {FLAT_CSV}")
print()
print(df.groupby(["league_name", "season"]).size().unstack(fill_value=0).to_string())

## 5. Filter to PL teams only

European/cup schedules include non-PL clubs. Filter so we only keep fixtures
involving at least one team that appeared in the PL that season.

In [ ]:
# Build set of PL teams per season
pl = df[df["league_name"] == "Premier League"]
pl_teams_by_season = (
    pd.concat([pl[["season", "home_team"]].rename(columns={"home_team": "team"}),
               pl[["season", "away_team"]].rename(columns={"away_team": "team"})])
    .drop_duplicates()
)

# Keep any fixture where home OR away team was in the PL that season
home_in_pl = df.merge(pl_teams_by_season, left_on=["season", "home_team"], right_on=["season", "team"], how="inner")
away_in_pl = df.merge(pl_teams_by_season, left_on=["season", "away_team"], right_on=["season", "team"], how="inner")

df_filtered = pd.concat([home_in_pl, away_in_pl]).drop(columns=["team"]).drop_duplicates()
df_filtered = df_filtered.sort_values("date").reset_index(drop=True)

filtered_csv = OUTPUT_DIR.parent / "fixtures_pl_teams.csv"
df_filtered.to_csv(filtered_csv, index=False)
print(f"PL-team fixtures: {len(df_filtered)} rows → {filtered_csv}")
print()
print(df_filtered.groupby(["league_name", "season"]).size().unstack(fill_value=0).to_string())

In [ ]:
driver.quit()
print("Browser closed")